# CReST: Computational Reproducibility Audit

Audits training time, inference efficiency, parameter counts, and the computing environment for the locked final runs.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Locate existing training logs and timing metadata

from pathlib import Path
import os

BASE_DIR = Path.cwd()

print("Current directory:", BASE_DIR)
print("=" * 100)

# Search for files that are likely to contain training/runtime information
patterns = [
    "*.json",
    "*.jsonl",
    "*.csv",
    "*.txt",
    "*.log",
    "*.pkl",
    "*.pt",
    "*.pth",
]

files = []
for pattern in patterns:
    files.extend(BASE_DIR.rglob(pattern))

# Prioritise filenames likely related to final training/checkpoints/logs
keywords = [
    "final", "fold", "train", "training", "history",
    "log", "manifest", "checkpoint", "metrics",
    "summary", "runtime", "time"
]

candidates = []
for f in files:
    name = f.name.lower()
    if any(k in name for k in keywords):
        candidates.append(f)

candidates = sorted(set(candidates))

print(f"Found {len(candidates)} potentially relevant files.\n")

for i, f in enumerate(candidates, 1):
    try:
        size_kb = f.stat().st_size / 1024
        mtime = f.stat().st_mtime
        print(f"{i:03d}. {f.relative_to(BASE_DIR)}  [{size_kb:,.1f} KB]")
    except Exception:
        print(f"{i:03d}. {f}")

In [ ]:
# Audit final-training files for timing information

from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"

print("=" * 100)
print("A. FINAL 7-FOLD TRAINING SUMMARY")
print("=" * 100)

summary_path = RESULTS_DIR / "crest_final_7fold_training_summary.csv"

if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
    print("File:", summary_path)
    print("\nColumns:")
    print(df_summary.columns.tolist())
    print("\nContent:")
    display(df_summary)
else:
    print("NOT FOUND:", summary_path)


print("\n" + "=" * 100)
print("B. PER-FOLD TRAINING SUMMARY JSON FILES")
print("=" * 100)

for fold in range(7):
    path = RESULTS_DIR / f"fold{fold}_training_summary.json"

    print(f"\n--- Fold {fold} ---")
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        print("Keys:", list(obj.keys()))
        print(json.dumps(obj, indent=2))
    else:
        print("NOT FOUND:", path)


print("\n" + "=" * 100)
print("C. PER-FOLD FINAL TRAINING HISTORY FILES")
print("=" * 100)

for fold in range(7):
    path = RESULTS_DIR / f"fold{fold}_final_training_history.csv"

    print(f"\n--- Fold {fold} ---")
    if path.exists():
        df = pd.read_csv(path)

        print("Columns:", df.columns.tolist())
        print("Shape:", df.shape)

        # Show only first/last rows to avoid excessive output
        display(df.head(2))
        display(df.tail(2))
    else:
        print("NOT FOUND:", path)

In [ ]:
# Compute exact final-training time by outer fold

from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"

rows = []

for fold in range(7):
    history_path = RESULTS_DIR / f"fold{fold}_final_training_history.csv"

    if not history_path.exists():
        raise FileNotFoundError(f"Missing: {history_path}")

    df = pd.read_csv(history_path)

    required = {"epoch", "epoch_seconds"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Fold {fold}: missing required columns: {sorted(missing)}"
        )

    # Full computational cost = all epochs actually executed
    total_seconds = df["epoch_seconds"].sum()
    mean_epoch_seconds = df["epoch_seconds"].mean()
    sd_epoch_seconds = df["epoch_seconds"].std(ddof=1)

    # Selected checkpoint information
    summary_path = RESULTS_DIR / f"fold{fold}_training_summary.json"

    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    best_epoch = int(summary["best_epoch"])
    epochs_run = int(summary["epochs_run"])

    # Time elapsed by the end of the selected checkpoint epoch.
    # Useful for audit, but NOT the primary reported training cost.
    time_to_best_seconds = (
        df.loc[df["epoch"] <= best_epoch, "epoch_seconds"].sum()
    )

    rows.append({
        "outer_fold": fold,
        "best_epoch": best_epoch,
        "epochs_run": epochs_run,
        "mean_epoch_seconds": mean_epoch_seconds,
        "sd_epoch_seconds": sd_epoch_seconds,
        "total_training_seconds": total_seconds,
        "total_training_minutes": total_seconds / 60,
        "total_training_hours": total_seconds / 3600,
        "time_to_best_seconds": time_to_best_seconds,
        "time_to_best_hours": time_to_best_seconds / 3600,
    })

timing_df = pd.DataFrame(rows)

print("=" * 100)
print("FINAL 7-FOLD TRAINING-TIME AUDIT")
print("=" * 100)

display(
    timing_df[
        [
            "outer_fold",
            "best_epoch",
            "epochs_run",
            "mean_epoch_seconds",
            "total_training_hours",
            "time_to_best_hours",
        ]
    ].round(3)
)

print("\n" + "=" * 100)
print("AGGREGATE COMPUTATIONAL COST")
print("=" * 100)

total_all_seconds = timing_df["total_training_seconds"].sum()
mean_fold_seconds = timing_df["total_training_seconds"].mean()
median_fold_seconds = timing_df["total_training_seconds"].median()
mean_epoch_seconds = (
    sum(
        pd.read_csv(
            RESULTS_DIR / f"fold{fold}_final_training_history.csv"
        )["epoch_seconds"].sum()
        for fold in range(7)
    )
    /
    sum(
        len(
            pd.read_csv(
                RESULTS_DIR / f"fold{fold}_final_training_history.csv"
            )
        )
        for fold in range(7)
    )
)

print(f"Total epochs executed   : {timing_df['epochs_run'].sum()}")
print(f"Total 7-fold time       : {total_all_seconds / 3600:.3f} hours")
print(f"Mean time per fold      : {mean_fold_seconds / 3600:.3f} hours")
print(f"Median time per fold    : {median_fold_seconds / 3600:.3f} hours")
print(f"Mean time per epoch     : {mean_epoch_seconds:.2f} s "
      f"({mean_epoch_seconds / 60:.2f} min)")

# Save audit output
out_csv = RESULTS_DIR / "crest_final_7fold_training_time_audit.csv"
timing_df.to_csv(out_csv, index=False)

print("\nSaved:")
print(out_csv)

FINAL 7-FOLD TRAINING-TIME AUDIT


,outer_fold,best_epoch,epochs_run,mean_epoch_seconds,total_training_hours,time_to_best_hours
0,0,18,20,961.678,5.343,4.821
1,1,19,20,955.901,5.311,5.067
2,2,19,20,972.015,5.400,5.139
3,3,11,15,964.270,4.018,2.943
4,4,20,20,948.661,5.270,5.270
5,5,6,10,947.993,2.633,1.559
6,6,17,20,933.189,5.184,4.407



AGGREGATE COMPUTATIONAL COST
Total epochs executed   : 125
Total 7-fold time       : 33.159 hours
Mean time per fold      : 4.737 hours
Median time per fold    : 5.270 hours
Mean time per epoch     : 954.98 s (15.92 min)

Saved:
PROJECT_ROOT\results\crest_final_7fold_training_time_audit.csv


In [ ]:
# Audit final test outputs for inference/runtime metadata

from pathlib import Path
import json

BASE_DIR = Path.cwd()
FINAL_TEST_DIR = BASE_DIR / "results" / "final_test_outputs"

timing_keywords = [
    "time",
    "runtime",
    "elapsed",
    "duration",
    "inference",
    "seconds",
    "latency",
    "student"
]

print("=" * 100)
print("FINAL TEST EVALUATION METADATA AUDIT")
print("=" * 100)

for fold in range(7):
    path = FINAL_TEST_DIR / f"fold_{fold}" / "test_evaluation_summary.json"

    print(f"\n{'-' * 100}")
    print(f"FOLD {fold}")
    print(f"{'-' * 100}")

    if not path.exists():
        print("NOT FOUND:", path)
        continue

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    print("File:", path)
    print("Keys:", list(obj.keys()))

    print("\nFull content:")
    print(json.dumps(obj, indent=2))

    print("\nPotential timing-related fields:")
    found = False

    for key, value in obj.items():
        if any(k in key.lower() for k in timing_keywords):
            print(f"  {key}: {value}")
            found = True

    if not found:
        print("  No obvious timing field found.")

In [ ]:
# Aggregate final-test inference efficiency across all seven folds

from pathlib import Path
import json
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()
FINAL_TEST_DIR = BASE_DIR / "results" / "final_test_outputs"
RESULTS_DIR = BASE_DIR / "results"

rows = []

for fold in range(7):
    path = FINAL_TEST_DIR / f"fold_{fold}" / "test_evaluation_summary.json"

    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    rows.append({
        "outer_fold": fold,
        "n_students": int(obj["n_students"]),
        "latent_inference_seconds": float(obj["latent_inference_seconds"]),
        "students_per_second": float(obj["students_per_second"]),
        "milliseconds_per_student": float(obj["milliseconds_per_student"]),
        "masked_evaluation_seconds": float(obj["masked_evaluation_seconds"]),
        "checkpoint_epoch": int(obj["checkpoint_epoch"]),
        "model_updated_on_test": bool(obj["model_updated_on_test"]),
    })

infer_df = pd.DataFrame(rows)

print("=" * 100)
print("FINAL 7-FOLD INFERENCE-TIME AUDIT")
print("=" * 100)

display(
    infer_df[
        [
            "outer_fold",
            "checkpoint_epoch",
            "n_students",
            "latent_inference_seconds",
            "students_per_second",
            "milliseconds_per_student",
            "model_updated_on_test",
        ]
    ].round(4)
)

# Pooled computational efficiency across all held-out students
total_students = infer_df["n_students"].sum()
total_inference_seconds = infer_df["latent_inference_seconds"].sum()

pooled_ms_per_student = (
    total_inference_seconds / total_students
) * 1000

pooled_students_per_second = (
    total_students / total_inference_seconds
)

# Fold-equal summaries
mean_fold_ms = infer_df["milliseconds_per_student"].mean()
sd_fold_ms = infer_df["milliseconds_per_student"].std(ddof=1)
median_fold_ms = infer_df["milliseconds_per_student"].median()

print("\n" + "=" * 100)
print("AGGREGATE INFERENCE EFFICIENCY")
print("=" * 100)

print(f"Total held-out students       : {total_students:,}")
print(f"Total latent inference time   : {total_inference_seconds:.3f} s")
print(f"Pooled inference cost         : {pooled_ms_per_student:.4f} ms/student")
print(f"Pooled throughput             : {pooled_students_per_second:.2f} students/s")
print(f"Fold-equal mean               : {mean_fold_ms:.4f} ms/student")
print(f"Fold-level SD                 : {sd_fold_ms:.4f} ms/student")
print(f"Fold-level median             : {median_fold_ms:.4f} ms/student")
print(
    f"Fold range                    : "
    f"{infer_df['milliseconds_per_student'].min():.4f}--"
    f"{infer_df['milliseconds_per_student'].max():.4f} ms/student"
)

print("\nAll test evaluations model-frozen:",
      (~infer_df["model_updated_on_test"]).all())

# Save detailed audit
out_csv = RESULTS_DIR / "crest_final_7fold_inference_time_audit.csv"
infer_df.to_csv(out_csv, index=False)

print("\nSaved:")
print(out_csv)

FINAL 7-FOLD INFERENCE-TIME AUDIT


,outer_fold,checkpoint_epoch,n_students,latent_inference_seconds,students_per_second,milliseconds_per_student,model_updated_on_test
0,0,18,21649,27.9244,775.2725,1.2899,False
1,1,19,20949,25.4145,824.2918,1.2132,False
2,2,19,20769,25.5892,811.6314,1.2321,False
3,3,11,20324,26.1069,778.4905,1.2845,False
4,4,20,19460,23.6299,823.5314,1.2143,False
5,5,6,19694,26.0544,755.8814,1.3230,False
6,6,17,19719,25.0179,788.1947,1.2687,False



AGGREGATE INFERENCE EFFICIENCY
Total held-out students       : 142,564
Total latent inference time   : 179.737 s
Pooled inference cost         : 1.2607 ms/student
Pooled throughput             : 793.18 students/s
Fold-equal mean               : 1.2608 ms/student
Fold-level SD                 : 0.0420 ms/student
Fold-level median             : 1.2687 ms/student
Fold range                    : 1.2132--1.3230 ms/student

All test evaluations model-frozen: True

Saved:
PROJECT_ROOT\results\crest_final_7fold_inference_time_audit.csv


In [ ]:
# Audit locked architecture, parameter count, and training metadata

from pathlib import Path
import json
import torch

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "final_7fold"

print("=" * 100)
print("A. LOCKED ARCHITECTURE METADATA")
print("=" * 100)

arch_path = RESULTS_DIR / "crest_final_architecture_lock.json"

if arch_path.exists():
    with open(arch_path, "r", encoding="utf-8") as f:
        arch = json.load(f)

    print("File:", arch_path)
    print("Keys:", list(arch.keys()))
    print(json.dumps(arch, indent=2))
else:
    print("NOT FOUND:", arch_path)


print("\n" + "=" * 100)
print("B. LOCKED TRAINING PROTOCOL")
print("=" * 100)

protocol_path = RESULTS_DIR / "crest_final_training_protocol.json"

if protocol_path.exists():
    with open(protocol_path, "r", encoding="utf-8") as f:
        protocol = json.load(f)

    print("File:", protocol_path)
    print("Keys:", list(protocol.keys()))
    print(json.dumps(protocol, indent=2))
else:
    print("NOT FOUND:", protocol_path)


print("\n" + "=" * 100)
print("C. FINAL CHECKPOINT STRUCTURE")
print("=" * 100)

for fold in range(7):
    ckpt_path = (
        CHECKPOINT_DIR /
        f"fold_{fold}" /
        "crest_final_best.pt"
    )

    print(f"\n--- Fold {fold} ---")

    if not ckpt_path.exists():
        print("NOT FOUND:", ckpt_path)
        continue

    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(ckpt, dict):
        print("Top-level keys:", list(ckpt.keys()))

        # Look for likely parameter-count / architecture fields
        for key, value in ckpt.items():
            key_lower = str(key).lower()

            if any(
                token in key_lower
                for token in [
                    "param",
                    "architecture",
                    "config",
                    "model_name",
                    "epoch"
                ]
            ):
                if not isinstance(value, dict):
                    print(f"{key}: {value}")
                else:
                    print(f"{key}: dict with {len(value)} entries")
    else:
        print("Checkpoint object type:", type(ckpt))

    # Only need detailed structure from fold 0
    if fold == 0:
        print("\nFold-0 checkpoint detailed inspection:")

        if isinstance(ckpt, dict):
            for key, value in ckpt.items():
                if isinstance(value, dict):
                    print(f"  {key}: dict ({len(value)} entries)")
                elif torch.is_tensor(value):
                    print(
                        f"  {key}: tensor, "
                        f"shape={tuple(value.shape)}"
                    )
                else:
                    print(
                        f"  {key}: "
                        f"{type(value).__name__}"
                    )

In [ ]:
# Verify parameter count directly from final checkpoints

from pathlib import Path
import torch
import pandas as pd

BASE_DIR = Path.cwd()
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "final_7fold"

rows = []

for fold in range(7):

    ckpt_path = (
        CHECKPOINT_DIR /
        f"fold_{fold}" /
        "crest_final_best.pt"
    )

    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False
    )

    model_state = ckpt["model_state_dict"]
    optimizer_state = ckpt["optimizer_state_dict"]

    # All tensor elements stored in the model state_dict.
    # This can include parameters and, for some architectures, buffers.
    state_dict_numel = sum(
        tensor.numel()
        for tensor in model_state.values()
        if torch.is_tensor(tensor)
    )

    # Adam/AdamW keeps exp_avg and exp_avg_sq for optimised parameters.
    # Summing one moment tensor per parameter gives an independent
    # checkpoint-based count of parameters actually handled by the optimiser.
    optimizer_parameter_numel = 0
    optimizer_parameter_tensors = 0

    for state in optimizer_state["state"].values():

        if "exp_avg" in state:
            optimizer_parameter_numel += state["exp_avg"].numel()
            optimizer_parameter_tensors += 1

    rows.append({
        "outer_fold": fold,
        "checkpoint_epoch": ckpt["epoch"],
        "state_dict_tensors": len(model_state),
        "state_dict_numel": state_dict_numel,
        "optimizer_parameter_tensors": optimizer_parameter_tensors,
        "optimizer_parameter_numel": optimizer_parameter_numel,
        "counts_match": (
            state_dict_numel == optimizer_parameter_numel
        ),
    })

param_df = pd.DataFrame(rows)

print("=" * 100)
print("FINAL CHECKPOINT PARAMETER-COUNT AUDIT")
print("=" * 100)

display(param_df)

print("\n" + "=" * 100)
print("CONSISTENCY CHECK")
print("=" * 100)

print(
    "Unique state_dict counts:",
    sorted(param_df["state_dict_numel"].unique())
)

print(
    "Unique optimizer-derived counts:",
    sorted(param_df["optimizer_parameter_numel"].unique())
)

print(
    "All seven folds identical:",
    param_df["optimizer_parameter_numel"].nunique() == 1
)

print(
    "State_dict and optimizer counts agree in all folds:",
    param_df["counts_match"].all()
)

if param_df["optimizer_parameter_numel"].nunique() == 1:
    verified_count = int(
        param_df["optimizer_parameter_numel"].iloc[0]
    )

    print(
        f"\nVerified trainable/optimised parameter count: "
        f"{verified_count:,}"
    )

# Save audit
out_csv = (
    BASE_DIR /
    "results" /
    "crest_final_parameter_count_audit.csv"
)

param_df.to_csv(out_csv, index=False)

print("\nSaved:")
print(out_csv)

FINAL CHECKPOINT PARAMETER-COUNT AUDIT


,outer_fold,checkpoint_epoch,state_dict_tensors,state_dict_numel,optimizer_parameter_tensors,optimizer_parameter_numel,counts_match
0,0,18,216,1578024,208,1577398,False
1,1,19,216,1578024,208,1577398,False
2,2,19,216,1578024,208,1577398,False
3,3,11,216,1578024,208,1577398,False
4,4,20,216,1578024,208,1577398,False
5,5,6,216,1578024,208,1577398,False
6,6,17,216,1578024,208,1577398,False



CONSISTENCY CHECK
Unique state_dict counts: [1578024]
Unique optimizer-derived counts: [1577398]
All seven folds identical: True
State_dict and optimizer counts agree in all folds: False

Verified trainable/optimised parameter count: 1,577,398

Saved:
PROJECT_ROOT\results\crest_final_parameter_count_audit.csv


In [ ]:
# Hardware and software environment audit

import platform
import sys
import os
from pathlib import Path

import torch
import pandas as pd

try:
    import psutil
except ImportError:
    psutil = None

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"

print("=" * 100)
print("HARDWARE AND SOFTWARE ENVIRONMENT AUDIT")
print("=" * 100)

# Software environment

python_version = sys.version.split()[0]
pytorch_version = torch.__version__
os_name = platform.system()
os_release = platform.release()
os_version = platform.version()
architecture = platform.machine()

# CPU information

cpu_name = platform.processor()

# platform.processor() can occasionally be blank on Windows
if not cpu_name:
    try:
        import subprocess
        result = subprocess.run(
            ["wmic", "cpu", "get", "name"],
            capture_output=True,
            text=True,
            check=False
        )
        lines = [
            line.strip()
            for line in result.stdout.splitlines()
            if line.strip() and "Name" not in line
        ]
        if lines:
            cpu_name = lines[0]
    except Exception:
        cpu_name = "Unavailable"

logical_cores = os.cpu_count()

if psutil is not None:
    physical_cores = psutil.cpu_count(logical=False)
    total_ram_gb = psutil.virtual_memory().total / (1024**3)
else:
    physical_cores = None
    total_ram_gb = None

# GPU / CUDA information

cuda_available = torch.cuda.is_available()
cuda_version = torch.version.cuda

if cuda_available:
    gpu_count = torch.cuda.device_count()
    gpu_names = [
        torch.cuda.get_device_name(i)
        for i in range(gpu_count)
    ]
else:
    gpu_count = 0
    gpu_names = []

# PyTorch execution information

torch_threads = torch.get_num_threads()
torch_interop_threads = torch.get_num_interop_threads()

environment = {
    "python_version": python_version,
    "pytorch_version": pytorch_version,
    "operating_system": f"{os_name} {os_release}",
    "os_version": os_version,
    "system_architecture": architecture,
    "cpu_model": cpu_name,
    "physical_cpu_cores": physical_cores,
    "logical_cpu_cores": logical_cores,
    "total_ram_gb": total_ram_gb,
    "torch_cpu_threads": torch_threads,
    "torch_interop_threads": torch_interop_threads,
    "cuda_available": cuda_available,
    "cuda_version": cuda_version,
    "gpu_count": gpu_count,
    "gpu_names": "; ".join(gpu_names) if gpu_names else "None",
}

env_df = pd.DataFrame(
    {
        "environment_field": environment.keys(),
        "value": environment.values(),
    }
)

display(env_df)

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

print(f"Python                  : {python_version}")
print(f"PyTorch                 : {pytorch_version}")
print(f"Operating system        : {os_name} {os_release}")
print(f"CPU                     : {cpu_name}")
print(f"Physical CPU cores      : {physical_cores}")
print(f"Logical CPU cores       : {logical_cores}")

if total_ram_gb is not None:
    print(f"Installed RAM           : {total_ram_gb:.2f} GB")

print(f"PyTorch CPU threads     : {torch_threads}")
print(f"CUDA available          : {cuda_available}")
print(f"CUDA version            : {cuda_version}")
print(f"GPU count               : {gpu_count}")

if gpu_names:
    print(f"GPU(s)                  : {', '.join(gpu_names)}")
else:
    print("GPU(s)                  : None")

# Save

out_csv = RESULTS_DIR / "crest_hardware_environment_audit.csv"
env_df.to_csv(out_csv, index=False)

print("\nSaved:")
print(out_csv)

HARDWARE AND SOFTWARE ENVIRONMENT AUDIT


,environment_field,value
0,python_version,3.11.10
1,pytorch_version,2.9.0+cpu
2,operating_system,Windows 10
3,os_version,10.0.19045
4,system_architecture,AMD64
5,cpu_model,"Intel64 Family 6 Model 94 Stepping 3, GenuineI..."
6,physical_cpu_cores,4
7,logical_cpu_cores,8
8,total_ram_gb,23.881405
9,torch_cpu_threads,4



SUMMARY
Python                  : 3.11.10
PyTorch                 : 2.9.0+cpu
Operating system        : Windows 10
CPU                     : Intel64 Family 6 Model 94 Stepping 3, GenuineIntel
Physical CPU cores      : 4
Logical CPU cores       : 8
Installed RAM           : 23.88 GB
PyTorch CPU threads     : 4
CUDA available          : False
CUDA version            : None
GPU count               : 0
GPU(s)                  : None

Saved:
PROJECT_ROOT\results\crest_hardware_environment_audit.csv


In [ ]:
# Resolve exact CPU model name on Windows

import subprocess

cpu_name = None

# Preferred: PowerShell / CIM
try:
    result = subprocess.run(
        [
            "powershell",
            "-NoProfile",
            "-Command",
            "(Get-CimInstance Win32_Processor).Name"
        ],
        capture_output=True,
        text=True,
        check=False
    )

    lines = [
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip()
    ]

    if lines:
        cpu_name = lines[0]

except Exception:
    pass

print("Exact CPU model:", cpu_name)

Exact CPU model: Intel(R) Core(TM) i7-6700HQ CPU @ 2.60GHz
